In [ ]:
# Inventory Replenishment Agent - Google Colab Version
# This notebook follows the same single-cell style as the previous assignment.
# It creates or loads CSV data, runs the agent, and prints the evaluation results.

# # Inventory Replenishment Agent
# This notebook implements a multi-SKU inventory replenishment agent. It generates or loads `sales.csv`, `inventory.csv`, and `params.csv`, forecasts demand with EWMA, calculates safety stock from forecast error, simulates purchase orders with lead times, and compares the agent with a fixed `(s, Q)` baseline.
# 
# **Learning goals covered:** data preparation, forecasting, safety stock, agent decisions, simulation, daily rationale logs, fill rate, stockouts, and total cost.

# ## Design clarity and trust
# **Agent role:** minimize stockouts and total operating cost by choosing an order quantity or waiting each day.
# 
# **Inputs:** recent demand history, current on-hand inventory, open purchase orders, lead time, minimum order quantity, service level, and cost parameters. **Output:** an order quantity or a transparent wait decision.
# 
# **Guardrails:** the policy respects minimum order quantities, accounts for lead time, uses service level to calculate safety stock, limits ordering to projected lead-time-plus-review demand plus safety stock, and explicitly weighs holding cost against stockout cost.
# 
# **Ethics and trust:** fill rate and stockouts measure customer service reliability; cost metrics support responsible use of working capital. Every action has a daily rationale so a human can audit why the agent ordered or waited.

import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

SEED = 7
rng = np.random.default_rng(SEED)
DATA_DIR = Path(".")

# If the CSV files are beside this notebook, the next cell loads them.
# The fallback generator creates reproducible demonstration data.


def generate_demo_data(data_dir=DATA_DIR, days=120):
    dates = pd.date_range("2026-01-01", periods=days, freq="D")
    profiles = {
        "SKU-A": {"base": 18, "trend": 0.04, "season": 3, "lead": 5, "min_qty": 60, "service": 0.95, "unit": 8.0, "hold": 0.12, "stockout": 5.0, "opening": 170},
        "SKU-B": {"base": 11, "trend": 0.02, "season": 2, "lead": 8, "min_qty": 45, "service": 0.90, "unit": 14.0, "hold": 0.18, "stockout": 8.0, "opening": 105},
        "SKU-C": {"base": 27, "trend": -0.01, "season": 5, "lead": 3, "min_qty": 90, "service": 0.98, "unit": 4.0, "hold": 0.08, "stockout": 3.5, "opening": 220},
    }
    sales_rows = []
    for sku, p in profiles.items():
        for i, date in enumerate(dates):
            weekly = p["season"] * math.sin(2 * math.pi * (i % 7) / 7)
            promo = 8 if sku == "SKU-A" and i in {26, 55, 86, 108} else 0
            qty = max(0, int(round(p["base"] + p["trend"] * i + weekly + promo + rng.normal(0, 2.2))))
            sales_rows.append({"date": date, "sku": sku, "qty_sold": qty})
    sales = pd.DataFrame(sales_rows)
    inventory = pd.DataFrame([{"sku": sku, "opening_stock": p["opening"]} for sku, p in profiles.items()])
    params = pd.DataFrame([{
        "sku": sku, "unit_cost": p["unit"], "holding_cost_per_day": p["hold"],
        "stockout_cost": p["stockout"], "lead_time_days": p["lead"],
        "min_order_qty": p["min_qty"], "service_level": p["service"]
    } for sku, p in profiles.items()])
    sales.to_csv(data_dir / "sales.csv", index=False)
    inventory.to_csv(data_dir / "inventory.csv", index=False)
    params.to_csv(data_dir / "params.csv", index=False)
    return sales, inventory, params

required = [DATA_DIR / f for f in ("sales.csv", "inventory.csv", "params.csv")]
if all(p.exists() for p in required):
    sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["date"])
    inventory = pd.read_csv(DATA_DIR / "inventory.csv")
    params = pd.read_csv(DATA_DIR / "params.csv")
else:
    sales, inventory, params = generate_demo_data()

sales = sales.sort_values(["sku", "date"]).reset_index(drop=True)
daily_demand = sales.groupby(["date", "sku"], as_index=False)["qty_sold"].sum()
print("Sales rows:", len(sales), "| SKUs:", sorted(sales.sku.unique()))
display(daily_demand.head())


# ## 1. EWMA forecast and safety stock
# The EWMA forecast gives more weight to recent observations. Safety stock uses the standard deviation of one-step-ahead training forecast errors and the normal z-value for each target service level.

Z_BY_SERVICE = {0.90: 1.282, 0.95: 1.645, 0.98: 2.054, 0.99: 2.326}
ALPHA = 0.30
TRAIN_DAYS = 60
REVIEW_PERIOD = 1

def z_value(service_level):
    return min(Z_BY_SERVICE, key=lambda x: abs(x - float(service_level)))

def ewma_one_step(values, alpha=ALPHA):
    if len(values) == 0:
        return 0.0
    forecast = float(values[0])
    for value in values[1:]:
        forecast = alpha * float(value) + (1 - alpha) * forecast
    return max(0.0, forecast)

def make_model_inputs(sales, params):
    model = {}
    first_date = sales.date.min()
    train_end = first_date + pd.Timedelta(days=TRAIN_DAYS - 1)
    for sku in params.sku:
        series = sales.loc[sales.sku.eq(sku)].sort_values("date").set_index("date")["qty_sold"]
        train = series.loc[series.index <= train_end].to_numpy(dtype=float)
        one_step = []
        for i in range(1, len(train)):
            one_step.append(ewma_one_step(train[:i]))
        err_std = float(np.std(np.asarray(train[1:]) - np.asarray(one_step), ddof=1)) if len(one_step) > 1 else 0.0
        row = params.loc[params.sku.eq(sku)].iloc[0]
        safety = z_value(row.service_level) * err_std * math.sqrt(row.lead_time_days)
        model[sku] = {"history": list(train), "error_std": err_std, "safety_stock": safety, "train_end": train_end}
    return model

model = make_model_inputs(sales, params)
model_summary = pd.DataFrame([{
    "sku": sku, "forecast_error_std": round(v["error_std"], 2),
    "safety_stock": round(v["safety_stock"], 2), "lead_time_days": int(params.loc[params.sku.eq(sku), "lead_time_days"].iloc[0])
} for sku, v in model.items()])
display(model_summary)


# ## 2. Simulation engine and agent policy
# The agent reviews inventory position (on-hand + pipeline − backlog). It projects demand across lead time plus one review day. If projected inventory would fall below zero, it orders up to projected demand plus safety stock, respecting the SKU minimum order quantity.

def forecast_next(history, horizon, alpha=ALPHA):
    level = ewma_one_step(history, alpha)
    return [level] * horizon

class InventorySimulator:
    def __init__(self, opening_stock, lead_time, holding_cost, stockout_cost, unit_cost, fixed_order_cost=0.0):
        self.on_hand = float(opening_stock)
        self.lead_time = int(lead_time)
        self.holding_cost = float(holding_cost)
        self.stockout_cost = float(stockout_cost)
        self.unit_cost = float(unit_cost)
        self.fixed_order_cost = float(fixed_order_cost)
        self.pipeline = defaultdict(float)
        self.log = []
        self.orders = []
    def step(self, date, demand, order_qty, rationale):
        arriving = self.pipeline.pop(date, 0.0)
        self.on_hand += arriving
        pre_sales = self.on_hand
        fulfilled = min(self.on_hand, float(demand))
        unmet = max(0.0, float(demand) - fulfilled)
        self.on_hand -= fulfilled
        if order_qty > 0:
            arrival_date = date + pd.Timedelta(days=self.lead_time)
            self.pipeline[arrival_date] += order_qty
            self.orders.append({"date": date, "qty": order_qty, "arrival_date": arrival_date})
        holding = self.holding_cost * self.on_hand
        stockout = self.stockout_cost * unmet
        order_cost = self.fixed_order_cost + self.unit_cost * order_qty if order_qty > 0 else 0.0
        self.log.append({"date": date, "demand": demand, "arrivals": arriving, "pre_sales_stock": pre_sales,
            "fulfilled": fulfilled, "unmet": unmet, "on_hand": self.on_hand, "pipeline_units": sum(self.pipeline.values()),
            "order_qty": order_qty, "rationale": rationale, "holding_cost": holding,
            "stockout_cost": stockout, "order_cost": order_cost})

def run_policy(sales, inventory, params, model, policy="agent"):
    start = sales.date.min() + pd.Timedelta(days=TRAIN_DAYS)
    horizon = sales.loc[sales.date >= start].sort_values(["date", "sku"])
    sims = {}
    histories = {sku: list(v["history"]) for sku, v in model.items()}
    for sku in params.sku:
        inv_row = inventory.loc[inventory.sku.eq(sku)].iloc[0]
        p = params.loc[params.sku.eq(sku)].iloc[0]
        sims[sku] = InventorySimulator(inv_row.opening_stock, p.lead_time_days, p.holding_cost_per_day, p.stockout_cost, p.unit_cost)
    for date, day in horizon.groupby("date"):
        for sku, row in day.set_index("sku").iterrows():
            p = params.loc[params.sku.eq(sku)].iloc[0]
            sim = sims[sku]
            inv_pos = sim.on_hand + sum(sim.pipeline.values())
            lead_review = int(p.lead_time_days + REVIEW_PERIOD)
            fc = forecast_next(histories[sku], lead_review)
            expected = sum(fc)
            ss = model[sku]["safety_stock"]
            target = expected + ss
            if policy == "agent":
                if inv_pos < target:
                    raw_qty = target - inv_pos
                    qty = max(float(p.min_order_qty), raw_qty)
                    qty = int(math.ceil(qty))
                    rationale = f"ORDER: position {inv_pos:.0f} below target {target:.0f}; demand over L+R={expected:.0f}, safety stock={ss:.0f}"
                else:
                    qty = 0
                    rationale = f"WAIT: position {inv_pos:.0f} covers target {target:.0f}; avoid excess holding cost"
            else:
                rop = expected + ss
                if inv_pos <= rop:
                    qty = int(max(float(p.min_order_qty), math.ceil(expected)))
                    rationale = f"BASELINE ORDER: position {inv_pos:.0f} at/below ROP {rop:.0f}"
                else:
                    qty = 0
                    rationale = f"BASELINE WAIT: position {inv_pos:.0f} above ROP {rop:.0f}"
            sim.step(date, row.qty_sold, qty, rationale)
            histories[sku].append(float(row.qty_sold))
    return sims

agent_sims = run_policy(sales, inventory, params, model, policy="agent")
baseline_sims = run_policy(sales, inventory, params, model, policy="baseline")


# ## 3. KPIs and successful outputs

def summarize(sims, policy):
    rows = []
    logs = []
    for sku, sim in sims.items():
        log = pd.DataFrame(sim.log)
        demand = log.demand.sum()
        fulfilled = log.fulfilled.sum()
        rows.append({
            "policy": policy, "sku": sku, "demand_units": int(demand),
            "fulfilled_units": int(fulfilled), "stockout_units": int(log.unmet.sum()),
            "fill_rate": round(fulfilled / demand if demand else 1.0, 4),
            "orders": int((log.order_qty > 0).sum()),
            "holding_cost": round(log.holding_cost.sum(), 2),
            "stockout_cost": round(log.stockout_cost.sum(), 2),
            "order_cost": round(log.order_cost.sum(), 2),
            "total_cost": round(log.holding_cost.sum() + log.stockout_cost.sum() + log.order_cost.sum(), 2)
        })
        log.insert(0, "policy", policy); log.insert(1, "sku", sku); logs.append(log)
    return pd.DataFrame(rows), pd.concat(logs, ignore_index=True)

agent_kpis, agent_log = summarize(agent_sims, "agent")
baseline_kpis, baseline_log = summarize(baseline_sims, "baseline")
kpis = pd.concat([baseline_kpis, agent_kpis], ignore_index=True)
display(kpis)
display(agent_log[["date", "sku", "order_qty", "on_hand", "unmet", "rationale"]].head(20))
print("\nFULL DAILY AGENT LOG\n")
print(agent_log[["date", "sku", "order_qty", "on_hand", "unmet", "rationale"]].to_string(index=False))
kpis.to_csv("kpi_results.csv", index=False)
agent_log.to_csv("agent_daily_log.csv", index=False)
print("Saved kpi_results.csv and agent_daily_log.csv")


# ## Interpretation and scaling note
# - **Fill rate** is fulfilled demand divided by total demand.
# - **Stockout units** are demand that could not be fulfilled on the day it occurred.
# - **Total cost** combines holding cost, stockout cost, and variable purchase cost.
# - The agent log records whether the agent ordered or waited and explains the decision using inventory position, projected demand, safety stock, and holding-cost considerations.
# 
# **Trade-off reflection:** increasing service level raises safety stock and holding cost but can reduce stockouts and protect customer trust. A lower service level reduces working capital but increases the risk and cost of unavailable products.
# 
# **Scaling note:** In production, this policy could run as a scheduled container job after daily sales and inventory data arrive. Monitoring should alert on fill rate below target, rising forecast error, repeated stockouts, stale data, or unusual order quantities. Reliability controls should validate schemas, handle missing dates or SKUs, retry transient data/API failures, and preserve an auditable decision log. Cost controls should impose an inventory-position cap, track working capital by SKU, and require human review for orders above a configurable threshold.
# 
# For submission, upload this notebook and the generated CSV files to GitHub, run all cells, and include the displayed KPI table and daily log as evidence of successful outputs.

